# Data Cleaning: Complete Guide

This notebook covers all essential data cleaning steps required for structured datasets.

## Table of Contents
1. [Missing Values](#missing-values)
2. [Duplicate Records](#duplicate-records)
3. [Outliers](#outliers)
4. [Data Type Conversion](#data-type-conversion)
5. [Inconsistent Data](#inconsistent-data)
6. [Normalization & Standardization](#normalization--standardization)
7. [Data Validation](#data-validation)
8. [Feature Engineering](#feature-engineering)

## 1. Missing Values {#missing-values}

### Understanding Missing Data
- **MCAR (Missing Completely At Random)**: No pattern
- **MAR (Missing At Random)**: Depends on other variables
- **MNAR (Missing Not At Random)**: Depends on the missing value itself

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create sample dataset with missing values
df = pd.DataFrame({
    'ID': range(1, 11),
    'Age': [25, 30, np.nan, 45, 35, np.nan, 28, 32, 40, 29],
    'Salary': [50000, 60000, 75000, np.nan, 65000, 80000, np.nan, 70000, 85000, 55000],
    'Department': ['HR', 'IT', 'HR', np.nan, 'IT', 'Finance', 'HR', np.nan, 'Finance', 'IT']
})

print("Original DataFrame:")
print(df)
print("\nMissing Values Count:")
print(df.isnull().sum())
print("\nMissing Values Percentage:")
print((df.isnull().sum() / len(df)) * 100)

### Strategies for Handling Missing Values

In [ ]:
# Strategy 1: Deletion (Use with caution)
df_dropna = df.dropna()
print("After dropping all missing values:")
print(f"Shape: {df_dropna.shape}")

# Strategy 2: Drop rows with missing in specific column
df_drop_age = df.dropna(subset=['Age'])
print(f"\nAfter dropping missing Age: {df_drop_age.shape}")

# Strategy 3: Forward Fill (for time series)
df_ffill = df.fillna(method='ffill')
print(f"\nForward fill:\n{df_ffill}")

# Strategy 4: Backward Fill
df_bfill = df.fillna(method='bfill')
print(f"\nBackward fill:\n{df_bfill}")

In [ ]:
# Strategy 5: Mean/Median/Mode Imputation
df_copy = df.copy()

# Numerical columns - use mean or median
df_copy['Age'].fillna(df_copy['Age'].median(), inplace=True)
df_copy['Salary'].fillna(df_copy['Salary'].mean(), inplace=True)

# Categorical columns - use mode
df_copy['Department'].fillna(df_copy['Department'].mode()[0], inplace=True)

print("After imputation:")
print(df_copy)
print(f"\nMissing values remaining: {df_copy.isnull().sum().sum()}")

In [ ]:
# Strategy 6: KNN Imputation (preserves relationships)
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=3)
df_knn = pd.DataFrame(
    imputer.fit_transform(df[['Age', 'Salary']]),
    columns=['Age', 'Salary']
)
print("After KNN Imputation:")
print(df_knn)

## 2. Duplicate Records {#duplicate-records}

In [ ]:
# Create dataset with duplicates
df_dup = pd.DataFrame({
    'ID': [1, 2, 2, 3, 4, 4, 4],
    'Name': ['Alice', 'Bob', 'Bob', 'Charlie', 'David', 'David', 'David'],
    'Score': [85, 90, 90, 78, 92, 92, 92]
})

print("Dataset with duplicates:")
print(df_dup)
print(f"\nDuplicate rows:")
print(df_dup[df_dup.duplicated(keep=False)])

In [ ]:
# Identify duplicates
print(f"Total duplicated rows: {df_dup.duplicated().sum()}")
print(f"Duplicated in specific columns: {df_dup.duplicated(subset=['ID']).sum()}")

# Remove duplicates - keep first occurrence
df_clean = df_dup.drop_duplicates()
print(f"\nAfter removing duplicates (keep='first'):")
print(df_clean)

# Keep last occurrence
df_clean_last = df_dup.drop_duplicates(keep='last')
print(f"\nAfter removing duplicates (keep='last'):")
print(df_clean_last)

# Remove all duplicates
df_clean_all = df_dup.drop_duplicates(keep=False)
print(f"\nAfter removing all duplicates (keep=False):")
print(df_clean_all)

## 3. Outliers {#outliers}

In [ ]:
# Create dataset with outliers
df_outliers = pd.DataFrame({
    'Score': [75, 80, 82, 78, 85, 88, 90, 92, 95, 1000],  # 1000 is outlier
    'Age': [25, 28, 30, 32, 35, 38, 40, 42, 45, 150]      # 150 is outlier
})

print("Dataset:")
print(df_outliers)
print(f"\nDescriptive Statistics:")
print(df_outliers.describe())

In [ ]:
# Method 1: IQR (Interquartile Range)
Q1 = df_outliers['Score'].quantile(0.25)
Q3 = df_outliers['Score'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR Method for Score:")
print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

outliers_iqr = df_outliers[(df_outliers['Score'] < lower_bound) | (df_outliers['Score'] > upper_bound)]
print(f"\nOutliers detected:")
print(outliers_iqr)

# Remove outliers
df_no_outliers = df_outliers[(df_outliers['Score'] >= lower_bound) & (df_outliers['Score'] <= upper_bound)]
print(f"\nAfter removing outliers:")
print(df_no_outliers)

In [ ]:
# Method 2: Z-Score
from scipy import stats

z_scores = np.abs(stats.zscore(df_outliers['Score']))
print(f"Z-Scores: {z_scores}")

# Typically, |z| > 3 indicates outlier
df_z_clean = df_outliers[z_scores < 3]
print(f"\nAfter Z-Score removal (|z| < 3):")
print(df_z_clean)

In [ ]:
# Method 3: Capping/Clipping
df_capped = df_outliers.copy()
df_capped['Score'] = df_capped['Score'].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)
print("After capping outliers:")
print(df_capped)

## 4. Data Type Conversion {#data-type-conversion}

In [ ]:
# Create mixed data types
df_types = pd.DataFrame({
    'Age': ['25', '30', '35', '40'],
    'Salary': ['50000.50', '60000.75', '70000.00', '80000.25'],
    'Active': ['True', 'False', 'True', 'True'],
    'JoinDate': ['2020-01-15', '2021-03-20', '2019-06-10', '2022-11-05']
})

print("Original DataFrame:")
print(df_types)
print(f"\nData types:\n{df_types.dtypes}")

In [ ]:
# Convert data types
df_types['Age'] = df_types['Age'].astype('int')
df_types['Salary'] = df_types['Salary'].astype('float')
df_types['Active'] = df_types['Active'].astype('bool')
df_types['JoinDate'] = pd.to_datetime(df_types['JoinDate'])

print("After type conversion:")
print(df_types)
print(f"\nNew data types:\n{df_types.dtypes}")

## 5. Inconsistent Data {#inconsistent-data}

In [ ]:
# Inconsistent data examples
df_inconsistent = pd.DataFrame({
    'Department': ['HR', 'hr', 'HR ', ' IT', 'it', 'Finance', 'finance'],
    'Gender': ['M', 'Male', 'm', 'F', 'Female', 'f', 'M'],
    'Status': ['Active', 'active', 'ACTIVE', 'Inactive', 'inactive']
})

print("Inconsistent Data:")
print(df_inconsistent)

In [ ]:
# Standardize text - convert to lowercase, strip whitespace
df_clean = df_inconsistent.copy()
df_clean['Department'] = df_clean['Department'].str.lower().str.strip()
df_clean['Status'] = df_clean['Status'].str.lower().str.strip()

# Map inconsistent values
gender_mapping = {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female'}
df_clean['Gender'] = df_clean['Gender'].str.lower().map(gender_mapping)

print("After standardization:")
print(df_clean)

## 6. Normalization & Standardization {#normalization--standardization}

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_scale = pd.DataFrame({
    'Feature1': [100, 200, 150, 300, 250],
    'Feature2': [1, 2, 1.5, 3, 2.5]
})

print("Original Data:")
print(df_scale)

In [ ]:
# Min-Max Normalization (0-1 range)
scaler_minmax = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler_minmax.fit_transform(df_scale),
    columns=df_scale.columns
)
print("Min-Max Normalized (0-1):")
print(df_normalized)

In [ ]:
# Standardization (Z-score normalization)
scaler_std = StandardScaler()
df_standardized = pd.DataFrame(
    scaler_std.fit_transform(df_scale),
    columns=df_scale.columns
)
print("Standardized (mean=0, std=1):")
print(df_standardized)
print(f"\nMean: {df_standardized.mean()}")
print(f"Std Dev: {df_standardized.std()}")

## 7. Data Validation {#data-validation}

In [ ]:
# Validation checks
df_validate = pd.DataFrame({
    'Age': [25, 150, -5, 30, 35],
    'Email': ['user@gmail.com', 'invalid_email', 'user@company.com', None, 'user@domain.co'],
    'Salary': [50000, 60000, -1000, 70000, 80000]
})

print("Data to validate:")
print(df_validate)

# Validation 1: Range checks
invalid_age = df_validate[(df_validate['Age'] < 18) | (df_validate['Age'] > 120)]
print(f"\nInvalid Ages: {len(invalid_age)}")
print(invalid_age)

# Validation 2: Positive values
invalid_salary = df_validate[df_validate['Salary'] < 0]
print(f"\nInvalid Salaries: {len(invalid_salary)}")
print(invalid_salary)

In [ ]:
# Validation 3: Email format
import re

def validate_email(email):
    if pd.isna(email):
        return False
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))

df_validate['Valid_Email'] = df_validate['Email'].apply(validate_email)
print("Email validation:")
print(df_validate[['Email', 'Valid_Email']])

## 8. Feature Engineering {#feature-engineering}

In [ ]:
# Create feature engineering dataset
df_feature = pd.DataFrame({
    'Age': [25, 30, 35, 40, 45],
    'JoinDate': pd.to_datetime(['2020-01-15', '2019-06-20', '2021-03-10', '2018-11-05', '2022-02-14']),
    'Salary': [50000, 60000, 70000, 80000, 90000]
})

print("Original Data:")
print(df_feature)

In [ ]:
# Feature 1: Age Groups
df_feature['AgeGroup'] = pd.cut(df_feature['Age'], 
                                 bins=[0, 25, 35, 45, 100],
                                 labels=['Young', 'Mid', 'Senior', 'Veteran'])

# Feature 2: Years of Service
df_feature['YearsOfService'] = (pd.Timestamp.now() - df_feature['JoinDate']).dt.days / 365.25

# Feature 3: Salary Range
df_feature['SalaryRange'] = pd.cut(df_feature['Salary'], 
                                    bins=[0, 55000, 75000, 100000],
                                    labels=['Low', 'Medium', 'High'])

# Feature 4: Salary per Year of Service
df_feature['SalaryPerYear'] = df_feature['Salary'] / df_feature['YearsOfService']

print("\nAfter Feature Engineering:")
print(df_feature)

## Summary

### Data Cleaning Checklist:
- [ ] Check for missing values and decide on strategy
- [ ] Identify and handle duplicates
- [ ] Detect and treat outliers
- [ ] Convert data types appropriately
- [ ] Standardize inconsistent values
- [ ] Normalize/Standardize numerical features
- [ ] Validate data against business rules
- [ ] Engineer new features if needed
- [ ] Document all transformations
- [ ] Create audit trail for reproducibility